# Worker del analizador de chats en Google Colab

Convierte este notebook en un nodo worker más de la plataforma. Se conecta a RabbitMQ en la nube (CloudAMQP, **con TLS**) y descarga los adjuntos de la laptop del coordinador por un túnel HTTPS.

**Antes de correrlo (una vez):**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU T4*** (opcional; sin GPU también funciona, pero más lento).
2. Panel izquierdo, ícono de llave (**Secretos**). Crear estos tres y activarles "Acceso al notebook":
   - `RABBIT_URL`: la URL `amqps://…` de CloudAMQP;
   - `WORKER_TOKEN`: el mismo de `distribuido/config.env` de la laptop;
   - `COORDINADOR_URL`: la dirección `https://….trycloudflare.com` que muestra `cloudflared` en la laptop.

   Así las credenciales nunca quedan escritas en el notebook.
3. *Entorno de ejecución → Ejecutar todas*. La última celda se queda corriendo: ese es el worker. Para detenerlo, usar el botón de parar.

⚠️ Procesar con estos workers **solo el dataset generado**, nunca chats reales.

In [ ]:
# 1. Código del proyecto
import os, subprocess
REPO = "https://github.com/SharonBlanco/Analizador-de-Chats-de-Whatsapp.git"
if not os.path.isdir("/content/proyecto"):
    !git clone -q {REPO} /content/proyecto
else:
    !git -C /content/proyecto pull -q
%cd /content/proyecto
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "Sin GPU: el worker usará la CPU" 

In [ ]:
# 2. Librerías (Colab ya trae PyTorch con CUDA y ffmpeg)
!pip install -q pika==1.3.2 pysentimiento ultralytics faster-whisper

In [ ]:
# 3. Configuración: credenciales desde los Secretos de Colab
from google.colab import userdata
for clave in ("RABBIT_URL", "WORKER_TOKEN", "COORDINADOR_URL"):
    os.environ[clave] = userdata.get(clave)

NOMBRE = "colab-1"                     # cambiar a colab-2 en el segundo worker
POOLS = "texto,imagen,audio,adjuntos"  # qué colas atiende este worker
os.environ.update(WORKER_ID=NOMBRE, HOST_NAME=NOMBRE, WORKER_POOLS=POOLS, PYTHONUNBUFFERED="1")
print("Worker", NOMBRE, "· pools:", POOLS, "· coordinador:", os.environ["COORDINADOR_URL"])

In [ ]:
# 4. Worker (se queda corriendo). El supervisor lo reinicia si se queda sin memoria (M1).
!python distribuido/worker/supervisor.py